In [1]:
# 1. Install necessary libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

from unsloth import FastLanguageModel
import torch

# 2. Load the base Llama-3 Model (compressed to 4-bit)
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# 4. Load the ChatDoctor Dataset
from datasets import load_dataset
dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split = "train[:5000]")

# 5. Format the data for Llama-3
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are Vitalis AI, a healthcare assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)

# 6. Set up the Trainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # ADDED THIS to fix the PicklingError!
    ),
)

# 7. Start Training!
print("Starting Training...")
trainer_stats = trainer.train()

# 8. Save your Custom Model
model.save_pretrained("vitalis_ai_lora")
tokenizer.save_pretrained("vitalis_ai_lora")
print("Model saved successfully!")


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-iwhnwslt/unsloth_dddda64b8c4c4a48a6bfb1769ac89b5c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-iwhnwslt/unsloth_dddda64b8c4c4a48a6bfb1769ac89b5c
  Resolved https://github.com/unslothai/unsloth.git to commit 72254e0a81fbb6717c5cbc7112eb8324cb74be33
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 26.5 MB/s eta 0:00:00
  

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

Starting Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.979670
2,4.065432
3,3.943474
4,3.905080
5,3.323586
6,2.922907
7,3.035568
8,2.985291
9,2.887345
10,2.719376


Unsloth: Restored added_tokens_decoder metadata in vitalis_ai_lora/tokenizer_config.json.


Model saved successfully!


In [2]:
# Insert your Hugging Face Token here:
HF_TOKEN = "REDACTED_HF_TOKEN"

# This will convert your custom model to GGUF format (Q4_K_M is the best balance of speed/size)
# and automatically upload it to your Hugging Face account!
model.push_to_hub_gguf(
    "Ritik0102/vitalis-ai", # REPLACE with your HF username
    tokenizer,
    quantization_method = "q4_k_m",
    token = HF_TOKEN,
)


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_gjw05fg1/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3-8b-instruct.Q4_K_M.gguf:   0%|          | 14.3MB / 4.92GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Ritik0102/vitalis-ai
Unsloth: Cleaning up temporary files...


'Ritik0102/vitalis-ai'